# Day 073 — Exercise 3: synthesize

**What you'll build:** `synthesize(text, voice, tts_fn, rate, pitch) -> bytes` — the core TTS function with `tts_fn=None` injection.

**Why it matters:** All higher-level functions delegate to `synthesize`. Getting the injection pattern and parameter forwarding right makes every downstream function automatically testable.

In [ ]:
import asyncio
_mock_tts = lambda text, **kw: b'AUDIO:' + text[:12].encode()


## Task

Implement `synthesize`:

1. If `tts_fn is not None`: `return tts_fn(text, voice=voice, rate=rate, pitch=pitch)`
2. `import edge_tts` (lazy)
3. Define `async def _run()` that creates `edge_tts.Communicate(text, voice, rate=rate, pitch=pitch)`, collects `chunk['data']` for chunks where `chunk['type'] == 'audio'`, returns `b''.join(chunks)`
4. `return asyncio.run(_run())`

## Your Implementation

In [ ]:
def synthesize(text: str, voice: str = 'en-US-AriaNeural',
               tts_fn=None, rate: str = '+0%', pitch: str = '+0Hz') -> bytes:
    """Synthesize text to audio bytes using Edge TTS.

    Args:
        text:   Text or SSML string
        voice:  Edge TTS voice ShortName
        tts_fn: callable(text, voice, rate, pitch) -> bytes for testing
        rate:   Speed adjustment e.g. '+10%', '-20%'
        pitch:  Pitch adjustment e.g. '+5Hz', '-2Hz'
    Returns:
        Audio bytes (MP3 when using real Edge TTS)
    """
    raise NotImplementedError


In [ ]:
def synthesize(text, voice='en-US-AriaNeural', tts_fn=None,
               rate='+0%', pitch='+0Hz'):
    if tts_fn is not None:
        return tts_fn(text, voice=voice, rate=rate, pitch=pitch)
    import edge_tts
    async def _run():
        comm = edge_tts.Communicate(text, voice, rate=rate, pitch=pitch)
        chunks = []
        async for chunk in comm.stream():
            if chunk['type'] == 'audio':
                chunks.append(chunk['data'])
        return b''.join(chunks)
    return asyncio.run(_run())


## Automated checks

In [ ]:

score, total = 0, 5
try:
    # returns bytes
    audio = synthesize('Hello world!', tts_fn=_mock_tts)
    assert isinstance(audio, bytes) and len(audio) > 0
    score += 1; print("✅ returns non-empty bytes")

    # tts_fn receives text as first arg
    captured = {}
    def _cap(text, **kw):
        captured['text'] = text; captured.update(kw)
        return b'captured'
    synthesize('Test sentence.', voice='en-GB-LibbyNeural',
               tts_fn=_cap, rate='+10%', pitch='+2Hz')
    assert captured.get('text') == 'Test sentence.'
    score += 1; print("✅ tts_fn receives text as first positional arg")

    # all kwargs forwarded
    assert captured.get('voice') == 'en-GB-LibbyNeural'
    assert captured.get('rate')  == '+10%'
    assert captured.get('pitch') == '+2Hz'
    score += 1; print("✅ voice/rate/pitch forwarded as kwargs to tts_fn")

    # different text → different bytes (mock encodes text prefix)
    a1 = synthesize('Hello there.', tts_fn=_mock_tts)
    a2 = synthesize('Goodbye now.', tts_fn=_mock_tts)
    assert a1 != a2, "Different text should give different audio bytes"
    score += 1; print("✅ different text produces different bytes")

    # default voice used when not specified
    captured2 = {}
    def _cap2(text, **kw): captured2.update(kw); return b'x'
    synthesize('Hi', tts_fn=_cap2)
    assert captured2.get('voice') == 'en-US-AriaNeural'
    score += 1; print("✅ default voice en-US-AriaNeural when voice not specified")

except Exception as e:
    print(f"❌ {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def synthesize(text, voice='en-US-AriaNeural', tts_fn=None,
               rate='+0%', pitch='+0Hz'):
    if tts_fn is not None:
        return tts_fn(text, voice=voice, rate=rate, pitch=pitch)
    import edge_tts
    async def _run():
        comm = edge_tts.Communicate(text, voice, rate=rate, pitch=pitch)
        chunks = []
        async for chunk in comm.stream():
            if chunk['type'] == 'audio':
                chunks.append(chunk['data'])
        return b''.join(chunks)
    return asyncio.run(_run())
```

**Why only collect `chunk['type'] == 'audio'` chunks?** The stream also yields `WordBoundary` chunks (timing data for word highlighting) and `SessionEnd`. Only the audio chunks contain the actual audio data.

</details>